[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab01.ipynb)

# Bab 1 — Pengantar Visi Komputer untuk Kesehatan
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Notebook ini berdiri sendiri dan dirancang untuk **Google Colab dengan GPU T4**
(*Runtime → Change runtime type → T4 GPU*), lalu **Runtime → Run all**.

Isi:
1. Info lingkungan (GPU, versi paket) dan pencatat waktu tiap sel penting.
2. Unduh enam subset **MedMNIST v2** berlisensi **CC BY 4.0** dari Zenodo (sumber resmi): PneumoniaMNIST (X-ray), OrganAMNIST (CT), BreastMNIST (USG), BloodMNIST (mikroskopi), RetinaMNIST (fundus), OCTMNIST (OCT).
3. Galeri modalitas, perbandingan resolusi 28/64/128/224 piksel, dan distribusi kelas.
4. Baseline pertama pada PneumoniaMNIST: regresi logistik piksel (28 px) vs CNN kecil (64 px) dengan sensitivitas, spesifisitas, PPV, NPV, AUC (+ CI 95 % bootstrap), matriks kebingungan, kurva ROC, dan contoh salah klasifikasi.
5. Data disimpan **permanen di Google Drive** (`MyDrive/Buku Visi Komputer Kesehatan/dataset/medmnist/`); bila sudah ada dan lengkap (ukuran + MD5 Zenodo), tidak diunduh ulang.
6. Semua hasil disimpan ke `/content/hasil_bab01/` lalu dizip menjadi `/content/hasil_bab01.zip`.

> **Perhatian.** Data MedMNIST sudah dianonimkan dan **tidak dimaksudkan untuk penggunaan klinis**. Model di notebook ini hanya untuk pembelajaran.
> Atribusi: Yang dkk. (2023), *Scientific Data*, 10, 41, https://doi.org/10.1038/s41597-022-01721-8 — data: https://doi.org/10.5281/zenodo.10519652.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`
Sel ini sama di semua notebook buku. Colab akan meminta izin akses Drive (akun Anda sendiri).
Dataset disimpan di `MyDrive/Buku Visi Komputer Kesehatan/dataset/<nama_dataset>/`; fungsi `pastikan_dataset(nama)`
memeriksa keberadaan, ukuran, dan MD5 berkas, lalu hanya mengunduh dari sumber resmi bila belum ada atau rusak.

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 1.0 Lingkungan dan pencatat waktu

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil
from pathlib import Path

T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB01_SMOKE") == "1"          # uji cepat di luar Colab (penulis); di Colab = False
OUT  = Path(os.environ.get("BAB01_OUT",  "/content/hasil_bab01"))
DATA = DATASET_DIR / "medmnist"                      # folder Drive permanen (dari sel standar)
OUT.mkdir(parents=True, exist_ok=True); DATA.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    WAKTU[nama] = round(dt, 2)
    print(f"[waktu] {nama}: {dt:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?)")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'")
    info_mem = jalankan("free -h | head -2")
    print(info_cpu, info_mem)
    (OUT / "info_lingkungan.txt").write_text(
        f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

## 1.1 Instalasi paket

In [ ]:
with catat_waktu("1_pip_install"):
    try:
        import medmnist
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist==3.0.2"], check=True)
        import medmnist
import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import sklearn
from medmnist import INFO
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

PERANGKAT = "cuda" if torch.cuda.is_available() else "cpu"
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
         "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
         "medmnist": medmnist.__version__, "numpy": np.__version__, "sklearn": sklearn.__version__,
         "matplotlib": matplotlib.__version__, "perangkat": PERANGKAT}
print(json.dumps(VERSI, indent=1))
plt.rcParams.update({"font.size": 10, "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight"})
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

## 1.2 Pastikan data ada di Drive, lalu muat (sumber resmi: Zenodo)
Semua subset di bawah berlisensi **CC BY 4.0** (dicek dari `INFO[flag]['license']`). Berkas `.npz` dibaca dari folder Drive
(`root=` folder Drive); pengunduhan hanya terjadi bila `.npz` belum ada atau tidak lolos cek ukuran/MD5.
Berkas `.npz` dibaca sekali ke memori, sehingga tidak perlu disalin ke disk lokal.

In [ ]:
MODALITAS = [  # (flag, modalitas, ukuran untuk galeri)
    ("pneumoniamnist", "X-ray dada", 128),
    ("organamnist",    "CT abdomen (irisan aksial)", 64),
    ("breastmnist",    "USG payudara", 128),
    ("bloodmnist",     "Mikroskopi sel darah", 64),
    ("retinamnist",    "Fundus retina", 128),
    ("octmnist",       "OCT retina", 64),
]
if SMOKE:
    MODALITAS = [(f, m, 28) for f, m, _ in MODALITAS]
RES_PNEU = [28, 128] if SMOKE else [28, 64, 128, 224]

NAMA_ID = {
 "pneumoniamnist": ["normal", "pneumonia"],
 "breastmnist":    ["ganas", "normal/jinak"],
 "octmnist":       ["CNV", "DME", "drusen", "normal"],
 "retinamnist":    ["DR 0", "DR 1", "DR 2", "DR 3", "DR 4"],
 "bloodmnist":     ["basofil", "eosinofil", "eritroblas", "granulosit imatur", "limfosit", "monosit", "neutrofil", "trombosit"],
 "organamnist":    ["kandung kemih", "femur kiri", "femur kanan", "jantung", "ginjal kiri", "ginjal kanan", "hati", "paru kiri", "paru kanan", "pankreas", "limpa"],
}

_SUDAH_CEK = set()
def muat(flag, size, split):
    fn = berkas_medmnist(flag, size)
    if (flag, size) not in _SUDAH_CEK:
        pastikan_dataset("medmnist", [fn]); _SUDAH_CEK.add((flag, size))
    kelas = getattr(medmnist, INFO[flag]["python_class"])
    ds = kelas(split=split, download=False, size=size, root=str(DATA))
    return ds.imgs, ds.labels.reshape(-1)

DATASET = {}
with catat_waktu("2_pastikan_dan_muat_data"):
    for flag, mod, size in MODALITAS:
        assert INFO[flag]["license"] == "CC BY 4.0", flag
        assert len(NAMA_ID[flag]) == len(INFO[flag]["label"]), flag
        DATASET[(flag, size)] = {s: muat(flag, size, s) for s in ("train", "val", "test")}
    for size in RES_PNEU:
        if ("pneumoniamnist", size) not in DATASET:
            DATASET[("pneumoniamnist", size)] = {s: muat("pneumoniamnist", size, s) for s in ("train", "val", "test")}
    if ("pneumoniamnist", 64) not in DATASET:          # dipakai CNN
        DATASET[("pneumoniamnist", 64)] = {s: muat("pneumoniamnist", 64, s) for s in ("train", "val", "test")}

baris = []
for flag, mod, size in MODALITAS:
    d = DATASET[(flag, size)]
    baris.append({"dataset": INFO[flag]["python_class"], "modalitas": mod, "tugas": INFO[flag]["task"],
                  "kelas": len(INFO[flag]["label"]), "latih": len(d["train"][1]), "validasi": len(d["val"][1]),
                  "uji": len(d["test"][1]), "bentuk_citra": "x".join(map(str, d["train"][0].shape[1:])),
                  "lisensi": INFO[flag]["license"]})
TABEL_DATASET = pd.DataFrame(baris)
TABEL_DATASET.to_csv(OUT / "tabel_1_1_dataset.csv", index=False)
TABEL_DATASET

## 1.3 Galeri enam modalitas (Gambar 1.1)

In [ ]:
def ambil_contoh(y, n=6, rng=np.random.default_rng(SEED)):
    # satu contoh per kelas lebih dulu, lalu acak sisanya
    pilih = []
    for k in np.unique(y):
        pilih.append(int(rng.choice(np.where(y == k)[0])))
        if len(pilih) == n: break
    sisa = np.setdiff1d(np.arange(len(y)), pilih)
    pilih += list(rng.choice(sisa, n - len(pilih), replace=False))
    return pilih

with catat_waktu("3_galeri_modalitas"):
    fig, ax = plt.subplots(len(MODALITAS), 6, figsize=(10.5, 1.95 * len(MODALITAS)))
    for i, (flag, mod, size) in enumerate(MODALITAS):
        X, y = DATASET[(flag, size)]["test"]
        for j, idx in enumerate(ambil_contoh(y)):
            a = ax[i, j]; img = X[idx]
            a.imshow(img, cmap="gray" if img.ndim == 2 else None)
            a.set_xticks([]); a.set_yticks([])
            a.set_title(NAMA_ID[flag][y[idx]], fontsize=8)
            if j == 0: a.set_ylabel(mod, fontsize=8.5)
    fig.suptitle("Contoh citra uji MedMNIST v2 (CC BY 4.0)", fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT / "gambar_1_1_galeri_modalitas.png"); plt.show()

## 1.4 Perbandingan resolusi PneumoniaMNIST (Gambar 1.2)

In [ ]:
with catat_waktu("4_resolusi"):
    Xs = {s: DATASET[("pneumoniamnist", s)]["test"] for s in RES_PNEU}
    y_t = Xs[RES_PNEU[0]][1]
    idx = [int(np.where(y_t == 0)[0][3]), int(np.where(y_t == 1)[0][3])]
    fig, ax = plt.subplots(2, len(RES_PNEU), figsize=(2.6 * len(RES_PNEU), 5.4))
    for r, i in enumerate(idx):
        for c, s in enumerate(RES_PNEU):
            ax[r, c].imshow(Xs[s][0][i], cmap="gray"); ax[r, c].set_xticks([]); ax[r, c].set_yticks([])
            ax[r, c].set_title(f"{s}×{s} px" + (" — " + NAMA_ID['pneumoniamnist'][y_t[i]] if c == 0 else ""), fontsize=9)
    fig.tight_layout(); fig.savefig(OUT / "gambar_1_2_resolusi_pneumonia.png"); plt.show()
    pd.DataFrame([{"ukuran_px": s, "MB_npz": round(sum(p.stat().st_size for p in DATA.glob(f"pneumoniamnist{'' if s==28 else '_'+str(s)}.npz"))/1e6, 1)} for s in RES_PNEU]).to_csv(OUT / "tabel_1_2_ukuran_resolusi.csv", index=False)

## 1.5 Distribusi kelas (Gambar 1.3, Tabel 1.3)

In [ ]:
with catat_waktu("5_distribusi_kelas"):
    rows = []
    for flag, mod, size in MODALITAS:
        for split in ("train", "val", "test"):
            y = DATASET[(flag, size)][split][1]
            for k, nama in enumerate(NAMA_ID[flag]):
                rows.append({"dataset": INFO[flag]["python_class"], "split": split, "kelas": nama, "jumlah": int((y == k).sum())})
    DIST = pd.DataFrame(rows); DIST.to_csv(OUT / "tabel_1_3_distribusi_kelas.csv", index=False)
    rasio = (DIST[DIST.split == "train"].groupby("dataset")["jumlah"].agg(lambda s: s.max() / max(s.min(), 1)).rename("rasio_mayoritas_minoritas"))
    rasio.round(1).to_csv(OUT / "tabel_1_3b_rasio_ketidakseimbangan.csv")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.6]})
    p = DIST[DIST.dataset == "PneumoniaMNIST"].pivot(index="split", columns="kelas", values="jumlah").loc[["train", "val", "test"]]
    p.index = ["latih", "validasi", "uji"]
    p.plot.bar(ax=ax[0], color=["#0B3D5C", "#F28C28"], rot=0); ax[0].set_title("PneumoniaMNIST per split"); ax[0].set_ylabel("jumlah citra"); ax[0].set_xlabel("")
    for c in ax[0].containers: ax[0].bar_label(c, fontsize=8)
    rasio.sort_values().plot.barh(ax=ax[1], color="#0B3D5C"); ax[1].set_title("Rasio kelas terbanyak : tersedikit (data latih)"); ax[1].set_xlabel("rasio"); ax[1].set_ylabel(""); ax[1].set_xlim(0, rasio.max() * 1.12)
    for c in ax[1].containers: ax[1].bar_label(c, fmt="%.1f", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "gambar_1_3_distribusi_kelas.png"); plt.show()
print(rasio.round(1))

## 1.6 Baseline A — regresi logistik pada piksel (28 px)
Setiap citra 28×28 diratakan menjadi 784 angka. Ambang keputusan 0,5.

In [ ]:
def metrik(y, p, thr=0.5, n_boot=1000, seed=SEED):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    rng = np.random.default_rng(seed); aucs = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2: aucs.append(roc_auc_score(y[i], p[i]))
    return {"n": int(len(y)), "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
            "akurasi": (tp + tn) / len(y), "sensitivitas": tp / (tp + fn), "spesifisitas": tn / (tn + fp),
            "PPV": tp / (tp + fp) if tp + fp else float("nan"), "NPV": tn / (tn + fn) if tn + fn else float("nan"),
            "AUC": roc_auc_score(y, p), "AUC_CI95": [float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))]}

with catat_waktu("6_baseline_regresi_logistik"):
    (Xtr, ytr), (Xva, yva), (Xte, yte) = [DATASET[("pneumoniamnist", 28)][s] for s in ("train", "val", "test")]
    f = lambda X: X.reshape(len(X), -1).astype(np.float32) / 255.0
    t0 = time.perf_counter()
    lr = LogisticRegression(max_iter=2000, C=1.0)
    lr.fit(f(Xtr), ytr)
    WAKTU["6a_latih_regresi_logistik"] = round(time.perf_counter() - t0, 2)
    p_lr = lr.predict_proba(f(Xte))[:, 1]
    M_LR = metrik(yte, p_lr)
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in M_LR.items()}, indent=1))

## 1.7 Baseline B — CNN kecil (64 px, GPU)
Empat blok konvolusi, 10 epoch, Adam, kerugian *binary cross-entropy*. Model terbaik dipilih dari **AUC validasi** (bukan data uji).
Tidak ada *flip* horizontal: pada foto toraks, letak jantung kiri–kanan bermakna klinis.

In [ ]:
class CNNKecil(nn.Module):
    def __init__(self):
        super().__init__()
        def blok(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.fitur = nn.Sequential(blok(1, 16), blok(16, 32), blok(32, 64), blok(64, 128))
        self.kepala = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, 1))
    def forward(self, x): return self.kepala(self.fitur(x)).squeeze(1)

def ke_tensor(X): return torch.tensor(X, dtype=torch.float32).unsqueeze(1).div(255.0).sub(0.5).div(0.5)

@torch.no_grad()
def prediksi(model, X, bs=512):
    model.eval(); out = []
    for i in range(0, len(X), bs): out.append(torch.sigmoid(model(X[i:i+bs].to(PERANGKAT))).cpu())
    return torch.cat(out).numpy()

EPOCH = 2 if SMOKE else 10
UKURAN_CNN = 28 if SMOKE else 64
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
(Xtr, ytr), (Xva, yva), (Xte, yte) = [DATASET[("pneumoniamnist", UKURAN_CNN)][s] for s in ("train", "val", "test")]
Ttr, Tva, Tte = ke_tensor(Xtr), ke_tensor(Xva), ke_tensor(Xte)
Ytr = torch.tensor(ytr, dtype=torch.float32)
model = CNNKecil().to(PERANGKAT)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
n_param = sum(p.numel() for p in model.parameters())
print("parameter:", n_param, "| perangkat:", PERANGKAT)

RIWAYAT, terbaik = [], (-1, None, None)
with catat_waktu("7_latih_cnn_total"):
    for ep in range(1, EPOCH + 1):
        t0 = time.perf_counter(); model.train(); urut = torch.randperm(len(Ttr)); tot = 0.0
        for i in range(0, len(Ttr), 128):
            b = urut[i:i+128]; x, yb = Ttr[b].to(PERANGKAT), Ytr[b].to(PERANGKAT)
            loss = F.binary_cross_entropy_with_logits(model(x), yb)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        auc_va = roc_auc_score(yva, prediksi(model, Tva))
        dt = time.perf_counter() - t0
        RIWAYAT.append({"epoch": ep, "loss_latih": tot / len(Ttr), "auc_validasi": auc_va, "detik": dt})
        print(f"epoch {ep:2d} | loss {tot/len(Ttr):.4f} | AUC val {auc_va:.4f} | {dt:.1f} s")
        if auc_va > terbaik[0]: terbaik = (auc_va, ep, {k: v.detach().clone() for k, v in model.state_dict().items()})
model.load_state_dict(terbaik[2])
p_cnn = prediksi(model, Tte)
M_CNN = metrik(yte, p_cnn)
pd.DataFrame(RIWAYAT).to_csv(OUT / "tabel_1_4_riwayat_latih_cnn.csv", index=False)
torch.save(model.state_dict(), OUT / "cnn_kecil_pneumonia_terbaik.pt")
print("epoch terbaik (AUC validasi):", terbaik[1])
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in M_CNN.items()}, indent=1))

## 1.8 Perbandingan baseline: tabel metrik, kurva belajar, ROC, matriks kebingungan (Tabel 1.5, Gambar 1.4–1.6)

In [ ]:
with catat_waktu("8_gambar_evaluasi"):
    TABEL_METRIK = pd.DataFrame([{"model": "Regresi logistik (28 px)", **M_LR}, {"model": f"CNN kecil ({UKURAN_CNN} px)", **M_CNN}])
    TABEL_METRIK.to_csv(OUT / "tabel_1_5_metrik_baseline.csv", index=False)

    R = pd.DataFrame(RIWAYAT)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].plot(R.epoch, R.loss_latih, "o-", color="#0B3D5C"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss latih"); ax[0].set_title("Kerugian data latih")
    ax[1].plot(R.epoch, R.auc_validasi, "o-", color="#F28C28"); ax[1].axvline(terbaik[1], ls="--", c="gray"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("AUC"); ax[1].set_title("AUC data validasi (garis putus = model terpilih)")
    fig.tight_layout(); fig.savefig(OUT / "gambar_1_4_kurva_belajar_cnn.png"); plt.show()

    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    for nama, p, c in [("Regresi logistik", p_lr, "#0B3D5C"), ("CNN kecil", p_cnn, "#F28C28")]:
        fpr, tpr, _ = roc_curve(yte, p); ax.plot(fpr, tpr, color=c, lw=2, label=f"{nama} (AUC {roc_auc_score(yte, p):.3f})")
    ax.plot([0, 1], [0, 1], "--", c="gray", lw=1); ax.set_xlabel("1 − spesifisitas"); ax.set_ylabel("sensitivitas"); ax.set_title("Kurva ROC — PneumoniaMNIST (uji)"); ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "gambar_1_5_roc.png"); plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.6))
    for a, (nama, M) in zip(ax, [("Regresi logistik", M_LR), ("CNN kecil", M_CNN)]):
        cm = np.array([[M["TN"], M["FP"]], [M["FN"], M["TP"]]])
        a.imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2): a.text(j, i, cm[i, j], ha="center", va="center", fontsize=12, color="white" if cm[i, j] > cm.max() / 2 else "black")
        a.set_xticks([0, 1], ["prediksi normal", "prediksi pneumonia"]); a.set_yticks([0, 1], ["normal", "pneumonia"]); a.set_title(nama)
    fig.tight_layout(); fig.savefig(OUT / "gambar_1_6_matriks_kebingungan.png"); plt.show()
TABEL_METRIK.round(3)

## 1.9 Contoh salah klasifikasi CNN (Gambar 1.7)

In [ ]:
with catat_waktu("9_salah_klasifikasi"):
    yhat = (p_cnn >= 0.5).astype(int)
    FPi = np.where((yhat == 1) & (yte == 0))[0]; FNi = np.where((yhat == 0) & (yte == 1))[0]
    FPi = FPi[np.argsort(-p_cnn[FPi])][:4]; FNi = FNi[np.argsort(p_cnn[FNi])][:4]
    fig, ax = plt.subplots(2, 4, figsize=(9, 4.8))
    for r, (judul, ids) in enumerate([("Positif palsu (normal → pneumonia)", FPi), ("Negatif palsu (pneumonia → normal)", FNi)]):
        for c in range(4):
            a = ax[r, c]; a.set_xticks([]); a.set_yticks([])
            if c < len(ids): a.imshow(Xte[ids[c]], cmap="gray"); a.set_title(f"p = {p_cnn[ids[c]]:.2f}", fontsize=9)
            if c == 0: a.set_ylabel(judul, fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / "gambar_1_7_salah_klasifikasi.png"); plt.show()

## 1.10 Ringkasan, simpan, dan zip

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {k: bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 4)
    if isinstance(o, np.integer): return int(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
RINGKASAN = bersih({"smoke": SMOKE, "versi": VERSI, "waktu_detik": WAKTU,
    "dataset": TABEL_DATASET.to_dict(orient="records"),
    "baseline_regresi_logistik_28px": M_LR, "baseline_cnn": {"ukuran_px": UKURAN_CNN, "epoch": EPOCH, "epoch_terbaik": terbaik[1], "parameter": n_param, **M_CNN},
    "riwayat_cnn": RIWAYAT})
(OUT / "ringkasan_bab01.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass